# 🛢️ Sprint 1: Adquisición y Validación de Datos
**Proyecto:** Análisis de la Dinámica Productiva en la Cuenca Cuyana  
**Objetivo de este notebook:** Cargar los datos crudos, explorar la estructura, aislar los registros de la Cuenca Cuyana (Mendoza), tratar valores faltantes y generar las primeras variables derivadas (como la fecha de primera producción).

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
# Configuración visual básica
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

# Definimos la ruta donde tu compañero guardó el archivo procesado de la auditoría
RUTA_DATOS = "../data/processed/produccion_pozos_cuenca_cuyana_2021_2026.csv"

## 1. Explorar columnas y diccionario de datos
Cargamos el dataset histórico consolidado y revisamos sus tipos de datos.

In [35]:
# Carga de datos
df = pd.read_csv(RUTA_DATOS)

# Exploración inicial
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
display(df.info())
display(df.describe().T)

Filas: 246240 | Columnas: 38
<class 'pandas.DataFrame'>
RangeIndex: 246240 entries, 0 to 246239
Data columns (total 38 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   idempresa               246240 non-null  str    
 1   anio                    246240 non-null  int64  
 2   mes                     246240 non-null  int64  
 3   idpozo                  246240 non-null  int64  
 4   prod_pet                246240 non-null  float64
 5   prod_gas                246240 non-null  float64
 6   prod_agua               246240 non-null  float64
 7   iny_agua                246240 non-null  float64
 8   iny_gas                 246240 non-null  float64
 9   iny_co2                 246240 non-null  float64
 10  iny_otro                246240 non-null  float64
 11  tef                     246240 non-null  float64
 12  vida_util               15525 non-null   float64
 13  tipoextraccion          246240 non-null  str    
 14  ti

None

,count,mean,std,min,25%,50%,75%,max
anio,246240.0,2023.317495,1.623234,2021.0,2022.0,2023.0,2025.00,2026.00
mes,246240.0,6.241309,3.415961,1.0,3.0,6.0,9.00,12.00
idpozo,246240.0,119576.223501,27702.114278,596.0,122646.0,123578.0,124519.00,167276.00
prod_pet,246240.0,22.100513,59.121913,0.0,0.0,0.0,0.00,1245.60
prod_gas,246240.0,1.042237,3.966390,0.0,0.0,0.0,0.00,657.22
prod_agua,246240.0,710.086137,2401.554691,0.0,0.0,0.0,0.00,28221.28
iny_agua,246240.0,705.662538,5571.317121,0.0,0.0,0.0,0.00,218837.80
iny_gas,246240.0,0.000000,0.000000,0.0,0.0,0.0,0.00,0.00
iny_co2,246240.0,0.000000,0.000000,0.0,0.0,0.0,0.00,0.00
iny_otro,246240.0,0.000000,0.000000,0.0,0.0,0.0,0.00,0.00


## 2. Filtrar e identificar registros (Foco en Mendoza)
Validamos que estamos trabajando con la Cuenca Cuyana y limitamos el alcance a la provincia de Mendoza para nuestro análisis.


In [36]:
# Validamos las cuencas y provincias disponibles
print("Distribución por Cuenca:\n", df['cuenca'].value_counts())
print("\nDistribución por Provincia:\n", df['provincia'].value_counts())

# Filtramos estrictamente Mendoza si hay pozos de otras provincias en la cuenca
df_mendoza = df[df['provincia'] == 'Mendoza'].copy()
print(f"\nPozos únicos en Mendoza: {df_mendoza['idpozo'].nunique()}")

Distribución por Cuenca:
 cuenca
CUYANA    246240
Name: count, dtype: int64

Distribución por Provincia:
 provincia
Mendoza    246240
Name: count, dtype: int64

Pozos únicos en Mendoza: 3702


## 3. Evaluar identificador único y continuidad mensual
Verificamos que `idpozo` sea consistente y calculamos cuántos meses de datos tiene cada pozo para saber si sirven para el análisis de maduración.

In [37]:
# Check de duplicados exactos por mes y pozo
duplicados = df_mendoza.duplicated(subset=['idpozo', 'anio', 'mes']).sum()
print(f"Registros duplicados por pozo-mes: {duplicados}")

# Continuidad: ¿Cuántos meses de registro tiene cada pozo?
meses_por_pozo = df_mendoza.groupby('idpozo').size()
display(meses_por_pozo.describe())

# Clasificamos la completitud
df_mendoza['meses_registrados'] = df_mendoza.groupby('idpozo')['idpozo'].transform('count')

Registros duplicados por pozo-mes: 0


count    3702.000000
mean       66.515397
std         4.770967
min         4.000000
25%        67.000000
50%        67.000000
75%        67.000000
max        68.000000
dtype: float64

## 4. Detectar fecha de primera producción
Buscamos el primer mes cronológico en el que el pozo reportó una producción de petróleo mayor a cero.

In [38]:
# Aseguramos que la fecha sea formato datetime
df_mendoza['fecha_data'] = pd.to_datetime(df_mendoza['fecha_data'])

# Filtramos solo los meses donde hubo producción real
df_prod_activa = df_mendoza[df_mendoza['prod_pet'] > 0]

# Buscamos la fecha mínima por pozo
fecha_primera_prod = df_prod_activa.groupby('idpozo')['fecha_data'].min().reset_index()
fecha_primera_prod.rename(columns={'fecha_data': 'fecha_primera_produccion'}, inplace=True)

# Unimos esta nueva variable al dataset original
df_mendoza = df_mendoza.merge(fecha_primera_prod, on='idpozo', how='left')
df_mendoza[['idpozo', 'fecha_data', 'prod_pet', 'fecha_primera_produccion']].head()

,idpozo,fecha_data,prod_pet,fecha_primera_produccion
0,162019,2021-01-31,0.00,NaT
1,160838,2021-01-31,91.23,2021-01-31
2,160267,2021-01-31,32.58,2021-01-31
3,160265,2021-01-31,843.53,2021-01-31
4,160142,2021-01-31,37.81,2021-01-31


## 5. Clasificación y Separación de Recursos
Analizamos la proporción de Convencional vs No Convencional y definimos columnas de interés.

In [39]:
# Proporción de tipo de recurso
print(df_mendoza['tipo_de_recurso'].value_counts(normalize=True) * 100)

# Para este proyecto, nos centraremos principalmente en:
columnas_clave = [
    'idpozo', 'fecha_data', 'anio', 'mes', 'areayacimiento', 
    'prod_pet', 'prod_gas', 'prod_agua', 'tipo_de_recurso', 
    'fecha_primera_produccion'
]
df_limpio = df_mendoza[columnas_clave].copy()

tipo_de_recurso
CONVENCIONAL    100.0
Name: proportion, dtype: float64


## 6. Reporte de Faltantes y Outliers
Evaluamos la calidad final de nuestro dataset reducido.

In [40]:
# Cuantificar nulos
print("Reporte de Valores Nulos:\n", df_limpio.isnull().sum())

# Detectar valores imposibles (ej. producción negativa)
prod_negativa = df_limpio[df_limpio['prod_pet'] < 0]
print(f"\nRegistros con producción de petróleo negativa: {len(prod_negativa)}")

# Limpieza básica de inconsistencias si existieran
df_limpio['prod_pet'] = df_limpio['prod_pet'].clip(lower=0) 
df_limpio['prod_gas'] = df_limpio['prod_gas'].clip(lower=0)

Reporte de Valores Nulos:
 idpozo                           0
fecha_data                       0
anio                             0
mes                              0
areayacimiento                   0
prod_pet                         0
prod_gas                         0
prod_agua                        0
tipo_de_recurso                  0
fecha_primera_produccion    172722
dtype: int64

Registros con producción de petróleo negativa: 0
